In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
import win32com.client as win32
import time  # Для измерения времени выполнения
import shutil
import re
from glob import glob
from datetime import timedelta
import gc

# Функция для форматирования времени в часы, минуты и секунды
def format_elapsed_time(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{int(hours)} часа(ов) {int(minutes)} минут(ы) {seconds:.2f} секунд"

# Функция для проверки, является ли файл скрытым (для Windows)
def is_hidden(file_path):
    try:
        # Получаем атрибуты файла
        file_attributes = os.stat(file_path).st_file_attributes
        # Проверяем, установлен ли флаг "скрытый"
        return file_attributes & 2 != 0  # 2 соответствует атрибуту "скрытый"
    except Exception:
        # Если возникла ошибка, считаем файл не скрытым
        return False

# Функция для форматирования даты в строковый формат 'YYYY-MM-DD'
def format_date_column(df, date_column):
    if date_column in df.columns:
        df[date_column] = pd.to_datetime(df[date_column], errors='coerce').dt.strftime('%Y-%m-%d')
    return df

# Функция для подключения к SQL Server с аутентификацией Windows
def connect_to_sql(server, database):
    connection_string = (
        f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server"
        "&trusted_connection=yes"
    )
    engine = create_engine(connection_string)
    return engine

# Функция для обработки ошибок и замены их на null
def handle_errors(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].apply(lambda x: None if isinstance(x, str) and x.strip() == '' else x)
    return df

In [2]:
FOLDER_PATH = os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!")
FOLDER_PATH_FEATURES = r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Дашбоард по рекламным кампаниям"
FOLDER_PATH_FOR_DB= os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям")
FOLDER_PATH_DUDL = os.path.normpath(r"\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ")

SQL_SERVER = "cl01sql"
SQL_DATABASE_DBREPORT = "DBReport"
SQL_DATABASE_DBPARTNERS = "DBPartners"

In [3]:
print("Начинаем собирать Базу Данных...")
start_all_time = time.time()
engine = connect_to_sql(SQL_SERVER, SQL_DATABASE_DBPARTNERS)

Начинаем собирать Базу Данных...


In [4]:
# 4. Получить данные из файла "Справочник.xlsx"
try:
    print("Начинаем получать данные для Справочника...")
    start_time = time.time()  # Запускаем таймер
    file_path_reference = os.path.join(FOLDER_PATH, "Справочник.xlsx")

    if os.path.exists(file_path_reference):
        # Список столбцов, которые нужно взять из файла
        columns_to_read = [
            "Артикул", "Артикул OZ", "Наименование", "Коллекция",
            "Бренд", "Размер", "Сезон", "Направление", "Розничный отдел",
            "Модель", "Группа", "Бизнес-группа", "Техсегмент",
            "Байер", "Две последние коллекции", "Основной артикул", "Ответственный за группу", "Себестоимость с НДС",
            "Процент выкупа", "НДС", "Группа для отчетов"
        ]

        # Типы данных для столбцов
        column_dtypes = {
            "Артикул": str,
            "Артикул OZ": str,
            "Наименование": str,
            "Коллекция": str,
            "Размер": str,
            "Бренд": str,
            "Сезон": str,
            "Направление": str,
            "Розничный отдел": str,
            "Модель": str,
            "Группа": str,
            "Бизнес-группа": str,
            "Техсегмент": str,
            "Байер": str,
            "Две последние коллекции": str,
            "Основной артикул": str,
            "Ответственный за группу": str,
            "Себестоимость с НДС": float,
            "Процент выкупа": float,
            "НДС": int,
            "Группа для отчетов": str
        }

        # Чтение файла с указанием нужных столбцов и типов данных
        df_reference = pd.read_excel(
            file_path_reference,
            sheet_name="Выгрузка для справочника",
            engine="openpyxl",
            usecols=columns_to_read,
            dtype=column_dtypes
        )

        # Удаление дубликатов
        df_reference = df_reference.drop_duplicates()

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Справочник:")
        print(df_reference.head())

        # Сохраняем результат
        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Справочника успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Справочник.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Справочника: {e}")

Начинаем получать данные для Справочника...
Первые 5 строк таблицы Справочник:
    Артикул               Наименование Размер Коллекция       Бренд  \
0  00001851  Балетки женские 8L2139-1C     36       NaN  T.TACCARDI   
1  00001851  Балетки женские 8L2139-1C     37       NaN  T.TACCARDI   
2  00001851  Балетки женские 8L2139-1C     38       NaN  T.TACCARDI   
3  00001851  Балетки женские 8L2139-1C     39       NaN  T.TACCARDI   
4  00001851  Балетки женские 8L2139-1C     40       NaN  T.TACCARDI   

             Сезон    Направление Розничный отдел     Модель Бизнес-группа  \
0  лето (закрытое)  Женская обувь   Женская обувь  8L2139-1C         Обувь   
1  лето (закрытое)  Женская обувь   Женская обувь  8L2139-1C         Обувь   
2  лето (закрытое)  Женская обувь   Женская обувь  8L2139-1C         Обувь   
3  лето (закрытое)  Женская обувь   Женская обувь  8L2139-1C         Обувь   
4  лето (закрытое)  Женская обувь   Женская обувь  8L2139-1C         Обувь   

   ... Техсегмент        

In [5]:
# 7. Создание таблицы "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу ВсегоРазмеров...")
    start_time = time.time()  # Запускаем таймер

    # Проверка наличия необходимых столбцов
    required_columns = ["Артикул", "Размер"]
    for col in required_columns:
        if col not in df_reference.columns:
            print(f"Ошибка: Отсутствует столбец '{col}' в df_reference.")
            exit()

    # Очищаем столбец "Размер":
    # - Преобразуем в строковый формат
    # - Удаляем лишние пробелы
    # - Заменяем пустые строки на None
    df_reference["Размер"] = df_reference["Размер"].astype(str).str.strip().replace('', None)

    # Создаем DataFrame с количеством размеров для каждого артикула
    df_reference_unique = (
        df_reference
        .drop_duplicates(subset=["Артикул", "Размер"])  # Удаляем дубликаты Артикул-Размер
        .groupby("Артикул")["Размер"]  # Группируем по артикулу
        .apply(lambda sizes: len(sizes.dropna().unique()) if len(sizes.dropna()) > 0 else 1)  # Подсчитываем размеры
        .reset_index(name="Всего размеров")
    )

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ВсегоРазмеров:")
    print(df_reference_unique.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ВсегоРазмеров успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ВсегоРазмеров: {e}")

Начинаем создавать таблицу ВсегоРазмеров...
Первые 5 строк таблицы ВсегоРазмеров:
    Артикул  Всего размеров
0  00001851               5
1  00001852               5
2  00001855               5
3  00001856               5
4  00001931               6
Таблица ВсегоРазмеров успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 37.16 секунд


In [6]:
# 6. Получить данные таблицы с SQL (РазмерыНаАгрегаторе)
try:
    print("Начинаем получать данные для РазмеровНаАгрегаторе...")
    start_time = time.time()  # Запускаем таймер
    query_sizes = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], COUNT(DISTINCT(a.[INVENTSIZEID])) AS [Колво размеров]
        FROM [DBPartners].[dbo].[WblmRepGetStockOzon] a
        WHERE [dt] >= '{(pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")}'
        GROUP BY [dt], [itemid]
    """
    df_sizes = pd.read_sql(query_sizes, engine)
    df_sizes = format_date_column(df_sizes, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы РазмерыНаАгрегаторе:")
    print(df_sizes.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для РазмеровНаАгрегаторе: {e}")

Начинаем получать данные для РазмеровНаАгрегаторе...
Первые 5 строк таблицы РазмерыНаАгрегаторе:
         Дата   Артикул  Колво размеров
0  2025-07-01  00006110               1
1  2025-07-01  00006140               1
2  2025-07-01  00006170               1
3  2025-07-01  00006220               1
4  2025-07-01  00006250               1
Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: 0 часа(ов) 1 минут(ы) 4.96 секунд


In [7]:
# 8. Связать "РазмерыНаАгрегаторе" с "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу Дистрибуция...")
    start_time = time.time()  # Запускаем таймер

    # Объединяем таблицы по полю "Артикул"
    df_distribution = pd.merge(df_sizes, df_reference_unique, on="Артикул", how="left")

    # Вычисляем дистрибуцию с проверкой на деление на ноль
    df_distribution["Дистрибуция"] = df_distribution.apply(
        lambda row: row["Колво размеров"] / row["Всего размеров"] if row["Всего размеров"] != 0 else 0,
        axis=1
    )

    # Оставляем только нужные столбцы
    df_distribution = df_distribution[["Дата", "Артикул", "Дистрибуция"]]

    # Форматирование даты
    df_distribution = format_date_column(df_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Дистрибуция:")
    print(df_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Дистрибуция успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Дистрибуция: {e}")

Начинаем создавать таблицу Дистрибуция...
Первые 5 строк таблицы Дистрибуция:
         Дата   Артикул  Дистрибуция
0  2025-07-01  00006110     0.166667
1  2025-07-01  00006140     0.166667
2  2025-07-01  00006170     0.166667
3  2025-07-01  00006220     0.166667
4  2025-07-01  00006250     0.166667
Таблица Дистрибуция успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 32.30 секунд


In [8]:
# 2. Получить данные таблицы с SQL (Остатки)
try:
    print("Начинаем получать данные для Остатков...")
    start_time = time.time()  # Запускаем таймер
    query_stock = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], SUM(a.[free_to_sell_amount]) AS [Остаток Агрегатора]
        FROM [DBPartners].[dbo].[WblmRepGetStockOzon] a
        WHERE [dt] >= '{(pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")}'
        GROUP BY [dt], [itemid]
    """
    df_stock = pd.read_sql(query_stock, engine)
    df_stock = format_date_column(df_stock, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатков:")
    print(df_stock.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для Остатков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для Остатков: {e}")

Начинаем получать данные для Остатков...
Первые 5 строк таблицы Остатков:
         Дата   Артикул  Остаток Агрегатора
0  2025-07-01  00006110                   0
1  2025-07-01  00006140                   0
2  2025-07-01  00006170                   0
3  2025-07-01  00006220                   0
4  2025-07-01  00006250                   0
Данные для Остатков успешно сохранены. Время выполнения: 0 часа(ов) 0 минут(ы) 53.26 секунд


In [9]:
# 9. Связать "Остатки" с "Дистрибуция"
try:
    print("Начинаем создавать таблицу Остатки с дистрибуцией...")
    start_time = time.time()  # Запускаем таймер
    df_stock_with_distribution = pd.merge(df_stock, df_distribution, on=["Дата", "Артикул"], how="left")
    df_stock_with_distribution = format_date_column(df_stock_with_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатки с дистрибуцией:")
    print(df_stock_with_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Остатки с дистрибуцией успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Остатки с дистрибуцией: {e}")

Начинаем создавать таблицу Остатки с дистрибуцией...
Первые 5 строк таблицы Остатки с дистрибуцией:
         Дата   Артикул  Остаток Агрегатора  Дистрибуция
0  2025-07-01  00006110                   0     0.166667
1  2025-07-01  00006140                   0     0.166667
2  2025-07-01  00006170                   0     0.166667
3  2025-07-01  00006220                   0     0.166667
4  2025-07-01  00006250                   0     0.166667
Таблица Остатки с дистрибуцией успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 4.42 секунд


In [10]:
del df_stock

In [11]:
# 3. Получить данные из файлов вложенной папки "Показатели по дням"
try:
    print("Начинаем получать данные для Воронки...")
    start_time = time.time()  # Запускаем таймер
    folder_path_weeks = os.path.join(FOLDER_PATH, "Показатели по дням")
    df_funnel = pd.DataFrame()

    if os.path.exists(folder_path_weeks):
        for file in os.listdir(folder_path_weeks):
            file_path = os.path.join(folder_path_weeks, file)

            # Пропускаем скрытые файлы
            if is_hidden(file_path):
                print(f"Пропущен скрытый файл: {file}")
                continue

            # Проверяем расширение файла
            if file.endswith((".xlsx", ".xls")):
                try:
                    # Список столбцов, которые нужно взять из файла
                    columns_to_read = [
                        "Дата",	"Артикул", "Показы, всего", "Показы на карточке товара", "Показы в поиске и каталоге",
                        "Позиция в поиске и каталоге", "В корзину, всего", "Заказано товаров", "Отменено товаров",
                        "Доставлено товаров", "Возвращено товаров", "Заказано на сумму", "В корзину из карточки товара",
                        "Выкупили ШТ", "ТипАктивности", "Расход, ₽", "Продажи, ₽", "Заказы, шт", "Показы", "Клики",  "Цена"
                    ]

                    if file.endswith(".xlsx"):
                        temp_df = pd.read_excel(file_path, sheet_name="Воронка", engine="openpyxl", usecols=columns_to_read)
                    elif file.endswith(".xls"):
                        temp_df = pd.read_excel(file_path, sheet_name="Воронка", engine="xlrd", usecols=columns_to_read)

                    # Переименование столбцов
                    temp_df.rename(columns={
                        "Артикул": "Артикул",
                        "Продажи, ₽": "Рекламные заказано на сумму",
                        "Показы": "Рекламные показы",
                        "Клики": "Рекламные показы на карточке товара",
                        "Заказы, шт": "Рекламные заказано товаров"
                    }, inplace=True, errors="ignore")

                    # Типы данных для столбцов
                    column_dtypes = {
                        "Артикул": str,
                        "Показы, всего": int,
                        "Показы на карточке товара": int,
                        "Показы в поиске и каталоге": int,
                        "Позиция в поиске и каталоге": float,
                        "В корзину, всего": int,
                        "Заказано товаров": int,
                        "Отменено товаров": int,
                        "Доставлено товаров": int,
                        "Возвращено товаров": int,
                        "Заказано на сумму": int,
                        "Выкупили ШТ": int,
                        "В корзину из карточки товара": int,
                        "ТипАктивности": str,
                        "Рекламные заказано на сумму": int,
                        "Рекламные заказано товаров": int,
                        "Рекламные показы на карточке товара": int,
                        "Рекламные показы": int,
                        "Расход, ₽": int,
                        "Цена": int
                    }

                    # Форматирование даты
                    temp_df = format_date_column(temp_df, 'Дата')

                    df_funnel = pd.concat([df_funnel, temp_df])
                except Exception as e:
                    print(f"Ошибка при чтении файла {file}: {e}")

        # Удаление лишних столбцов (если они остались)
        df_funnel = df_funnel[[
             "Дата",	"Артикул", "Показы, всего", "Показы на карточке товара", "Показы в поиске и каталоге",
                        "Позиция в поиске и каталоге", "В корзину, всего", "Заказано товаров", "Отменено товаров",
                        "Доставлено товаров", "Возвращено товаров", "Заказано на сумму", "В корзину из карточки товара",
                        "Выкупили ШТ", "ТипАктивности", "Расход, ₽", "Рекламные заказано на сумму", "Рекламные заказано товаров",
                        "Рекламные показы", "Рекламные показы на карточке товара", "Цена"
        ]]
        mask = pd.to_numeric(df_funnel['Расход, ₽'], errors='coerce').eq(0)
        df_funnel.loc[mask, 'ТипАктивности'] = 'Органика'
        
        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Воронка:")
        print(df_funnel.head())

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Воронки успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Папка 'Показатели по дням' не найдена.")
except Exception as e:
    print(f"Ошибка при получении данных для Воронки: {e}")

Начинаем получать данные для Воронки...
Первые 5 строк таблицы Воронка:
         Дата   Артикул  Показы, всего  Показы на карточке товара  \
0  2025-08-01  M6106140           9522                        361   
1  2025-08-01  W2088627          15493                        507   
2  2025-08-01  W1358261          14445                        348   
3  2025-08-01  M6106138          34809                        844   
4  2025-08-01  W1358259           6490                        149   

   Показы в поиске и каталоге  Позиция в поиске и каталоге  В корзину, всего  \
0                         857                        38.13                21   
1                        6587                        37.59                53   
2                        7132                        39.88                27   
3                        1674                        12.00                34   
4                        1929                       131.56                10   

   Заказано товаров  Отменено то

In [12]:
df_funnel

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Заказано на сумму,В корзину из карточки товара,Выкупили ШТ,ТипАктивности,"Расход, ₽",Рекламные заказано на сумму,Рекламные заказано товаров,Рекламные показы,Рекламные показы на карточке товара,Цена
0,2025-08-01,M6106140,9522,361,857,38.13,21,5,1,0,...,16865,21,4,Трафарет,556.81,0.0,0.0,7079.0,111.0,3373.0
1,2025-08-01,W2088627,15493,507,6587,37.59,53,16,4,8,...,36320,52,9,Трафарет,2815.54,2567.0,1.0,8665.0,168.0,2650.0
2,2025-08-01,W1358261,14445,348,7132,39.88,27,7,4,3,...,7875,27,2,Трафарет,3080.45,1125.0,1.0,12205.0,219.0,1125.0
3,2025-08-01,M6106138,34809,844,1674,12.00,34,11,5,2,...,41932,33,6,Трафарет,2397.55,0.0,0.0,27064.0,478.0,3975.0
4,2025-08-01,W1358259,6490,149,1929,131.56,10,4,2,4,...,5762,9,2,Трафарет,155.36,1404.0,1.0,3486.0,76.0,1477.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48566,2025-08-31,GW5609,3,0,0,NaN,0,0,0,0,...,0,0,0,Органика,NaN,NaN,NaN,NaN,NaN,NaN
48567,2025-08-31,J469QX000BUC9999,2,0,1,131.00,0,0,0,0,...,0,0,0,Органика,NaN,NaN,NaN,NaN,NaN,NaN
48568,2025-08-31,JH7174,1,0,0,NaN,0,0,0,0,...,0,0,0,Органика,NaN,NaN,NaN,NaN,NaN,NaN
48569,2025-08-31,50534694_977,1,0,0,NaN,0,0,0,0,...,0,0,0,Органика,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Путь до папки
folder_path_weeks = os.path.join(FOLDER_PATH, "Затраты", "Озон. Затраты из Аналитики New Format")

# Собираем все .xlsx файлы
files = glob(os.path.join(folder_path_weeks, "*.xlsx"))

df_list = []
for file in files:
    # --- достаём дату из названия файла ---
    filename = os.path.basename(file)  # например: "Аналитика продвижения_16.09.2025.xlsx"
    date_str = filename.split("_")[-1].replace(".xlsx", "")  # "16.09.2025"
    date_parsed = (pd.to_datetime(date_str, format="%d.%m.%Y") - timedelta(days=1)).strftime("%Y-%m-%d")

    # читаем, пропуская первую строку
    df_tmp = pd.read_excel(file, skiprows=1)

    # оставляем только нужные колонки
    cols_keep = ["SKU", "ID кампании", "Инструмент", "Место размещения"]
    df_tmp = df_tmp[cols_keep]

    # добавляем колонку "Дата"
    df_tmp["Дата"] = date_parsed

    df_list.append(df_tmp)

# объединяем все файлы
df_all = pd.concat(df_list, ignore_index=True)
df_all.rename(columns={'SKU': "Артикул OZ"},inplace=True)
df_all["Артикул OZ"] = df_all["Артикул OZ"].astype(str)

In [14]:
df_all

,Артикул OZ,ID кампании,Инструмент,Место размещения,Дата
0,841850783,17708398,Оплата за клик,Поиск и рекомендации,2025-09-30
1,1628955757,17330786,Оплата за клик,Поиск и рекомендации,2025-09-30
2,2366656419,17576553,Оплата за клик,Поиск и рекомендации,2025-09-30
3,1151435994,17577580,Оплата за клик,Поиск и рекомендации,2025-09-30
4,1066494382,17558878,Оплата за клик,Поиск и рекомендации,2025-09-30
...,...,...,...,...,...
1244168,1074035251,17588062,Оплата за клик,NaN,2025-09-29
1244169,1354181180,17559714,Оплата за клик,NaN,2025-09-29
1244170,2364307698,17007727,Оплата за клик,NaN,2025-09-29
1244171,989131265,17146731,Оплата за клик,NaN,2025-09-29


In [15]:
df_funnel.columns

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 'ТипАктивности',
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена'],
      dtype='object')

In [16]:
# 10. Связать "Воронка" с "Справочник"
try:
    print("Начинаем создавать таблицу ВоронкаСправочник...")
    start_time = time.time()  # Запускаем таймер

    # Проверка наличия необходимых столбцов
    required_columns = ["Дата", "Артикул"]
    for col in required_columns:
        if col not in df_funnel.columns:
            print(f"Ошибка: Отсутствует столбец '{col}' в df_funnel.")
            exit()

    # # Список столбцов, которые нужно взять из справочника
    # reference_columns = [
    #     "Артикул", "Артикул OZ", "Наименование", "Коллекция", "Бренд", "Сезон", "Направление",
    #     "Розничный отдел", "Модель", "Группа", "Бизнес-группа", "Техсегмент",
    #     "Байер", "Две последние коллекции", "Основной артикул", "Себестоимость с НДС",
    #     "Процент выкупа", "НДС", "Ответственный за группу", "Группа для отчетов"
    # ]

    # # Фильтруем справочник, оставляя только нужные столбцы
    # df_reference_filtered = df_reference[reference_columns]

    # # Приводим типы данных к строковому формату
    # df_funnel["Артикул"] = df_funnel["Артикул"].fillna('').astype(str).str[:8]  # Заменяем NaN на пустые строки
    # df_reference_filtered["Артикул"] = df_reference_filtered["Артикул"].fillna('').astype(str).str[:8]

    # # Объединение таблиц
    # df_funnel_reference = pd.merge(
    #     df_funnel,
    #     df_reference_filtered,
    #     left_on="Артикул",
    #     right_on="Артикул",
    #     how="left"
    # )

    # # Удаление дубликатов
    # df_funnel_reference = df_funnel_reference.drop_duplicates()

    # # Удаление лишних столбцов (если они остались)
    # # df_funnel_reference = df_funnel_reference.drop(columns=["Артикул WB"], errors="ignore")

    # Список столбцов, которые нужно взять из справочника
    reference_columns = [
        "Артикул", "Артикул OZ", "Наименование", "Коллекция", "Бренд", "Сезон", "Направление",
        "Розничный отдел", "Модель", "Группа", "Бизнес-группа", "Техсегмент",
        "Байер", "Две последние коллекции", "Основной артикул", "Себестоимость с НДС",
        "Процент выкупа", "НДС", "Ответственный за группу", "Группа для отчетов"
    ]

    # Фильтруем справочник
    df_reference_filtered = df_reference[reference_columns].copy()

    # Приводим типы данных к строке (обрезаем до 8 символов)
    df_funnel["Артикул"] = df_funnel["Артикул"].fillna('').astype(str).str[:8]
    df_reference_filtered["Артикул"] = df_reference_filtered["Артикул"].fillna('').astype(str).str[:8]

    # Удаляем дубликаты в справочнике по "Артикул" (сохраняем первую запись)
    df_reference_filtered = (
        df_reference_filtered
        .sort_values(["Артикул", "Артикул OZ"], na_position="last")
        .drop_duplicates(subset=["Артикул"], keep="first")
    )

    # Объединение
    df_funnel_reference = pd.merge(
        df_funnel,
        df_reference_filtered,
        on="Артикул",
        how="left"
    )

    # Удаление дубликатов "на всякий случай"
    df_funnel_reference = df_funnel_reference.drop_duplicates(ignore_index=True)

    # Форматирование даты
    df_funnel_reference = format_date_column(df_funnel_reference, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ВоронкаСправочник:")
    print(df_funnel_reference.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ВоронкаСправочник успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ВоронкаСправочник: {e}")

Начинаем создавать таблицу ВоронкаСправочник...
Первые 5 строк таблицы ВоронкаСправочник:
         Дата   Артикул  Показы, всего  Показы на карточке товара  \
0  2025-08-01  M6106140           9522                        361   
1  2025-08-01  W2088627          15493                        507   
2  2025-08-01  W1358261          14445                        348   
3  2025-08-01  M6106138          34809                        844   
4  2025-08-01  W1358259           6490                        149   

   Показы в поиске и каталоге  Позиция в поиске и каталоге  В корзину, всего  \
0                         857                        38.13                21   
1                        6587                        37.59                53   
2                        7132                        39.88                27   
3                        1674                        12.00                34   
4                        1929                       131.56                10   

   Заказано то

In [17]:
df_funnel_reference

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов
0,2025-08-01,M6106140,9522,361,857,38.13,21,5,1,0,...,Обувь,flat (L),Коновалова А.,2024SS,M6106140,1601.4616,0.956522,20.0,Жилин Даниил,Обувь
1,2025-08-01,W2088627,15493,507,6587,37.59,53,16,4,8,...,Обувь,high (AM),Коновалова А.,2025SS,W2088627,599.6341,0.928205,20.0,Жукова Наталья,Обувь
2,2025-08-01,W1358261,14445,348,7132,39.88,27,7,4,3,...,Обувь,flat (AM),Коновалова А.,2025SS,W1358261,345.7665,0.900000,20.0,Жукова Наталья,Обувь
3,2025-08-01,M6106138,34809,844,1674,12.00,34,11,5,2,...,Обувь,flat (L),Коновалова А.,2024SS,M6106138,1629.0899,0.771429,20.0,Жилин Даниил,Обувь
4,2025-08-01,W1358259,6490,149,1929,131.56,10,4,2,4,...,Обувь,flat (AM),Коновалова А.,2025SS,W1358259,350.9056,0.920000,20.0,Жукова Наталья,Обувь
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2956743,2025-08-31,GW5609,3,0,0,NaN,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2956744,2025-08-31,J469QX00,2,0,1,131.00,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2956745,2025-08-31,JH7174,1,0,0,NaN,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2956746,2025-08-31,50534694,1,0,0,NaN,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# del df_funnel

In [19]:
df_all['Артикул OZ']

0           841850783
1          1628955757
2          2366656419
3          1151435994
4          1066494382
              ...    
1244168    1074035251
1244169    1354181180
1244170    2364307698
1244171     989131265
1244172    1407480757
Name: Артикул OZ, Length: 1244173, dtype: object

In [20]:
df_funnel_reference['Артикул OZ']

0          1437702044
1          1899225035
2          1899441363
3          1437702191
4          1899441373
              ...    
2956743           NaN
2956744           NaN
2956745           NaN
2956746           NaN
2956747           NaN
Name: Артикул OZ, Length: 2956748, dtype: object

In [21]:
# объединяем с df_funnel
df_result = df_funnel_reference.merge(
    df_all,
    on=["Дата", "Артикул OZ"],
    how="left"
)

In [22]:
# Трафарет
mask = (
    ((df_result['Инструмент'] == 'Оплата за клик') &
    (df_result['Место размещения'] == 'Поиск и рекомендации')) |
    (df_result['ТипАктивности'] == 'Оплата за клик')
)
df_result.loc[mask, 'ТипАктивности'] = 'Трафарет'

# Вывод в топ
mask = (
    ((df_result['Инструмент'] == 'Оплата за клик') & 
     (df_result['Место размещения'] == 'Поиск')) |
    (df_result['ТипАктивности'] == 'ТОП')
)
df_result.loc[mask, 'ТипАктивности'] = 'Вывод в топ'

In [23]:
df_result['ТипАктивности'].unique()

array(['Трафарет', 'Органика', 'Оплата за заказ', 'Вывод в топ'],
      dtype=object)

In [24]:
# df_funnel_reference_copy = df_funnel_reference.copy()
df_funnel_reference = df_result

In [25]:
df_funnel_reference

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов,ID кампании,Инструмент,Место размещения
0,2025-08-01,M6106140,9522,361,857,38.13,21,5,1,0,...,2024SS,M6106140,1601.4616,0.956522,20.0,Жилин Даниил,Обувь,NaN,NaN,NaN
1,2025-08-01,W2088627,15493,507,6587,37.59,53,16,4,8,...,2025SS,W2088627,599.6341,0.928205,20.0,Жукова Наталья,Обувь,NaN,NaN,NaN
2,2025-08-01,W1358261,14445,348,7132,39.88,27,7,4,3,...,2025SS,W1358261,345.7665,0.900000,20.0,Жукова Наталья,Обувь,NaN,NaN,NaN
3,2025-08-01,M6106138,34809,844,1674,12.00,34,11,5,2,...,2024SS,M6106138,1629.0899,0.771429,20.0,Жилин Даниил,Обувь,NaN,NaN,NaN
4,2025-08-01,W1358259,6490,149,1929,131.56,10,4,2,4,...,2025SS,W1358259,350.9056,0.920000,20.0,Жукова Наталья,Обувь,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2966750,2025-08-31,GW5609,3,0,0,NaN,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2966751,2025-08-31,J469QX00,2,0,1,131.00,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2966752,2025-08-31,JH7174,1,0,0,NaN,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2966753,2025-08-31,50534694,1,0,0,NaN,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# 11. Связать "ВоронкаСправочник" с "Остатки с дистрибуцией"
try:
    print("Начинаем создавать таблицу ДБбезПризнаков...")
    start_time = time.time()  # Запускаем таймер
    df_final_db = pd.merge(df_funnel_reference, df_stock_with_distribution, left_on=["Дата", "Артикул"], right_on=["Дата", "Артикул"], how="left")
    df_final_db = format_date_column(df_final_db, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБбезПризнаков:")
    print(df_final_db.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБбезПризнаков успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБбезПризнаков: {e}")

Начинаем создавать таблицу ДБбезПризнаков...
Первые 5 строк таблицы ДБбезПризнаков:
         Дата   Артикул  Показы, всего  Показы на карточке товара  \
0  2025-08-01  M6106140           9522                        361   
1  2025-08-01  W2088627          15493                        507   
2  2025-08-01  W1358261          14445                        348   
3  2025-08-01  M6106138          34809                        844   
4  2025-08-01  W1358259           6490                        149   

   Показы в поиске и каталоге  Позиция в поиске и каталоге  В корзину, всего  \
0                         857                        38.13                21   
1                        6587                        37.59                53   
2                        7132                        39.88                27   
3                        1674                        12.00                34   
4                        1929                       131.56                10   

   Заказано товаров 

In [27]:
df_final_db

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов,ID кампании,Инструмент,Место размещения,Остаток Агрегатора,Дистрибуция
0,2025-08-01,M6106140,9522,361,857,38.13,21,5,1,0,...,1601.4616,0.956522,20.0,Жилин Даниил,Обувь,NaN,NaN,NaN,83.0,0.833333
1,2025-08-01,W2088627,15493,507,6587,37.59,53,16,4,8,...,599.6341,0.928205,20.0,Жукова Наталья,Обувь,NaN,NaN,NaN,169.0,1.000000
2,2025-08-01,W1358261,14445,348,7132,39.88,27,7,4,3,...,345.7665,0.900000,20.0,Жукова Наталья,Обувь,NaN,NaN,NaN,75.0,1.000000
3,2025-08-01,M6106138,34809,844,1674,12.00,34,11,5,2,...,1629.0899,0.771429,20.0,Жилин Даниил,Обувь,NaN,NaN,NaN,64.0,0.833333
4,2025-08-01,W1358259,6490,149,1929,131.56,10,4,2,4,...,350.9056,0.920000,20.0,Жукова Наталья,Обувь,NaN,NaN,NaN,228.0,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2966750,2025-08-31,GW5609,3,0,0,NaN,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2966751,2025-08-31,J469QX00,2,0,1,131.00,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2966752,2025-08-31,JH7174,1,0,0,NaN,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2966753,2025-08-31,50534694,1,0,0,NaN,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
# del df_funnel_reference, df_stock_with_distribution
# gc.collect()

In [29]:
# 5. Получить данные из файла !!!_Признаки для артикула и даты для Озон
try:
    print("Начинаем получать данные для Признаков...")
    start_time = time.time()  # Запускаем таймер
    file_path_features = os.path.join(FOLDER_PATH_FEATURES, "!!!_Признаки для артикула и даты для Озон.xlsx")
    if os.path.exists(file_path_features):
        df_item_features = pd.read_excel(file_path_features, sheet_name="Признаки для артикула", dtype=str, engine="openpyxl")
        df_date_features = pd.read_excel(file_path_features, sheet_name="Признаки для дат", dtype={0: "datetime64[ns]", **{i: str for i in range(1, 6)}}, engine="openpyxl")

        # Обработка ошибок
        df_item_features = handle_errors(df_item_features)
        df_date_features = handle_errors(df_date_features)

        # Форматирование даты
        df_date_features = format_date_column(df_date_features, 'Дата')

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки артикула:")
        print(df_item_features.head())

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки дат:")
        print(df_date_features.head())

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Признаков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл '!!!_Признаки для артикула и даты для Озон.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Признаков: {e}")

Начинаем получать данные для Признаков...
Первые 5 строк таблицы Признаки артикула:
    Артикул Признак Артикула 1 Признак Артикула 2 Признак Артикула 3  \
0  00001851                NaN                NaN                NaN   
1  00001852                NaN                NaN                NaN   
2  00001855                NaN                NaN                NaN   
3  00001856                NaN                NaN                NaN   
4  00001931                NaN                NaN                NaN   

  Признак Артикула 4 Признак Артикула 5  
0                NaN                NaN  
1                NaN                NaN  
2                NaN                NaN  
3                NaN                NaN  
4                NaN                NaN  
Первые 5 строк таблицы Признаки дат:
Empty DataFrame
Columns: [Дата, Признак Даты 1, Признак Даты 2, Признак Даты 3, Признак Даты 4, Признак Даты 5]
Index: []
Данные для Признаков успешно сохранены. Время выполнения: 0 часа(ов) 0 м

In [30]:
# 12. Связать "ДБбезПризнаков" с "Признаки для артикула"
try:
    print("Начинаем создавать таблицу ДБсПризнакамиАртикула...")
    start_time = time.time()  # Запускаем таймер
    df_final_db_item_features = pd.merge(df_final_db, df_item_features, left_on=["Артикул"], right_on=["Артикул"], how="left")
    df_final_db_item_features = format_date_column(df_final_db_item_features, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБсПризнакамиАртикула:")
    print(df_final_db_item_features.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБсПризнакамиАртикула успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнакамиАртикула: {e}")

Начинаем создавать таблицу ДБсПризнакамиАртикула...
Первые 5 строк таблицы ДБсПризнакамиАртикула:
         Дата   Артикул  Показы, всего  Показы на карточке товара  \
0  2025-08-01  M6106140           9522                        361   
1  2025-08-01  W2088627          15493                        507   
2  2025-08-01  W1358261          14445                        348   
3  2025-08-01  M6106138          34809                        844   
4  2025-08-01  W1358259           6490                        149   

   Показы в поиске и каталоге  Позиция в поиске и каталоге  В корзину, всего  \
0                         857                        38.13                21   
1                        6587                        37.59                53   
2                        7132                        39.88                27   
3                        1674                        12.00                34   
4                        1929                       131.56                10   

   Зак

In [31]:
del df_item_features

In [32]:
# 13. Связать "ДБсПризнакамиАртикула" с "Признаки для дат"
try:
    print("Начинаем создавать таблицу ДБсПризнаками...")
    start_time = time.time()
    df_final_db_all_features = pd.merge(df_final_db_item_features, df_date_features, on="Дата", how="left")
    df_final_db_all_features = format_date_column(df_final_db_all_features, 'Дата')

    print("Первые 5 строк таблицы ДБсПризнаками:")
    print(df_final_db_all_features.head())

    elapsed_time = time.time() - start_time
    print(f"Таблица ДБсПризнаками успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнаками: {e}")

Начинаем создавать таблицу ДБсПризнаками...
Первые 5 строк таблицы ДБсПризнаками:
         Дата   Артикул  Показы, всего  Показы на карточке товара  \
0  2025-08-01  M6106140           9522                        361   
1  2025-08-01  W2088627          15493                        507   
2  2025-08-01  W1358261          14445                        348   
3  2025-08-01  M6106138          34809                        844   
4  2025-08-01  W1358259           6490                        149   

   Показы в поиске и каталоге  Позиция в поиске и каталоге  В корзину, всего  \
0                         857                        38.13                21   
1                        6587                        37.59                53   
2                        7132                        39.88                27   
3                        1674                        12.00                34   
4                        1929                       131.56                10   

   Заказано товаров  О

In [ ]:
import pyarrow as pa
import pyarrow.csv as csv
table = pa.Table.from_pandas(df_final_db_all_features)

In [ ]:
# df_final_db_all_features.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon.csv"), index=False)
csv.write_csv(table, os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon.csv"))


In [ ]:
len(df_final_db_all_features.columns)

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 'ТипАктивности',
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена', 'Артикул OZ',
       'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление',
       'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент',
       'Байер', 'Две последние коллекции', 'Основной артикул',
       'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов', 'ID кампании',
       'Инструмент', 'Место размещения', 'Остаток Агрегатора', 'Дистрибуция',
       'Признак Артикула 1', 'Признак Артикула 2', 'Признак А

In [35]:
# del df_date_features
# gc.collect()

In [36]:
# # 14. Обработка ТОП 10%
# try:
#     print("Начинаем создавать таблицу с признаком ТОП 10%...")
#     start_time = time.time()

#     # Загрузка данных из Excel файла
#     file_path_top_10 = r"Z:\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!\ТОП 10% из ДБ.xlsx"
#     df_top_10 = pd.read_excel(file_path_top_10, sheet_name="Лист1", header=None)

#     # Пропускаем первые 9 строк
#     df_top_10 = df_top_10.iloc[9:]

#     # Явно указываем нужные столбцы (по индексам)
#     df_top_10 = df_top_10.iloc[:, [0, 1, 5]]  # Берем только столбцы 0, 1 и 5

#     # Переименовываем столбцы
#     df_top_10.columns = ["Группа", "АртикулKari", "Column6"]

#     # Фильтрация строк, где значение в столбце "Группа" не является пустым
#     df_top_10 = df_top_10[df_top_10["Группа"].notnull() & (df_top_10["Группа"] != "") & (df_top_10["Группа"] != "(пусто)")]

#     # Преобразование типа для столбца "Column6"
#     df_top_10["Column6"] = df_top_10["Column6"].astype(str)

#     # Переименование столбца "Column6" в "ТОП 10%"
#     df_top_10.rename(columns={"Column6": "ТОП 10%"}, inplace=True)

#     # Дополнительная фильтрация для столбца "ТОП 10%"
#     df_top_10 = df_top_10[df_top_10["ТОП 10%"].notnull() & (df_top_10["ТОП 10%"] != "")]

#     # Объединяем итоговую таблицу с данными ТОП 10%
#     df_final_db_all_features_with_top_10 = pd.merge(
#         df_final_db_all_features,
#         df_top_10,
#         left_on=["Группа", "Артикул"],
#         right_on=["Группа", "АртикулKari"],
#         how="left"
#     )

#     print("Первые 5 строк финальной таблицы с признаком ТОП 10%:")
#     print(df_final_db_all_features_with_top_10.head())

#     # Сохранение финальной таблицы
#     df_final_db_all_features_with_top_10.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнакамиИТОП10_Ozon.csv"), index=False)

#     elapsed_time = time.time() - start_time
#     print(f"Таблица с признаком ТОП 10% успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
# except Exception as e:
#     print(f"Ошибка при создании таблицы с признаком ТОП 10%: {e}")

In [37]:
# _df_final_db_all_features_with_top_10 = df_final_db_all_features_with_top_10[df_final_db_all_features_with_top_10['Дата'] >= '2025-09-04']

In [38]:
# _df_final_db_all_features_with_top_10['ТипАктивности'].unique()

In [39]:
# Функция для обновления Excel-файла с циклом попыток
def update_and_save_excel(file_path, new_file_path):
    max_attempts = 10  # Максимальное количество попыток
    attempt = 0

    while attempt < max_attempts:
        attempt += 1
        print(f"Попытка {attempt} обновить файл '{os.path.basename(file_path)}'...")

        try:
            # Открываем Excel приложение
            excel = win32.Dispatch("Excel.Application")
            excel.DisplayAlerts = False  # Отключает предупреждения Excel

            try:
                # Открываем книгу
                workbook = excel.Workbooks.Open(file_path)

                # Выполняем обновление всех данных (эквивалентно "Обновить всё" в Excel)
                print("Выполняем обновление данных...")
                workbook.RefreshAll()
                excel.CalculateUntilAsyncQueriesDone()  # Дожидаемся завершения обновления

                # Сохраняем оригинальный файл в FOLDER_PATH_FOR_DB
                workbook.SaveAs(file_path)
                print(f"Файл успешно сохранен с оригинальным именем в '{os.path.dirname(file_path)}'.")

                # Сохраняем файл с новым именем в FOLDER_PATH_FEATURES
                workbook.SaveAs(new_file_path)
                print(f"Файл успешно сохранен как '{os.path.basename(new_file_path)}'.")

                return True  # Успешное завершение

            except Exception as e:
                print(f"Ошибка при обновлении или сохранении файла: {e}")
            finally:
                # Закрываем книгу и выходим из Excel
                if 'workbook' in locals():
                    workbook.Close(SaveChanges=False)
                excel.Quit()

        except Exception as e:
            print(f"Ошибка при работе с Excel: {e}")

        # Если произошла ошибка, ждем перед следующей попыткой
        if attempt < max_attempts:
            print(f"Пауза перед следующей попыткой ({attempt + 1}/{max_attempts})...")
            time.sleep(60)  # Пауза 5 секунд

    return False  # Все попытки завершились неудачно

In [40]:
# 19. Обновить файл "Показы и затраты ОЗ_2.0.xlsx"
try:
    print("Подготовка данных для ДБ завершена.")
    # input("Начать обновление файлов ДБ? Для подтверждения нажмите Enter...")
    print("Начинаем обновлять файл 'Показы и затраты ОЗ_2.0.xlsx'...")
    start_time = time.time()  # Запускаем таймер

    # Путь к исходному файлу
    file_path_shows_expenses = os.path.join(FOLDER_PATH_FOR_DB, "Показы и затраты ОЗ_2.0.xlsx")

    if os.path.exists(file_path_shows_expenses):
        # Создаем новое имя файла с текущей датой без года
        current_month_day = time.strftime("%d.%m")  # Текущая дата в формате ДД.ММ
        new_file_name = f"Показы и затраты ОЗ_2.0 {current_month_day}.xlsx"
        new_file_path = os.path.join(FOLDER_PATH_FEATURES, new_file_name)

        # Путь для сохранения в дополнительную папку FOLDER_PATH_DUDL
        dudl_file_path = os.path.join(FOLDER_PATH_DUDL, new_file_name)

        # Удаляем старые файлы из FOLDER_PATH_DUDL
        try:
            if os.path.exists(FOLDER_PATH_DUDL):
                for filename in os.listdir(FOLDER_PATH_DUDL):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ОЗ_2\.0 (\d{2}\.\d{2})\.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_DUDL, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_DUDL}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_DUDL}': {delete_error}")

        # Удаляем старые файлы из FOLDER_PATH_FEATURES
        try:
            if os.path.exists(FOLDER_PATH_FEATURES):
                for filename in os.listdir(FOLDER_PATH_FEATURES):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ОЗ_2\.0 (\d{2}\.\d{2})\.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_FEATURES, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_FEATURES}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_FEATURES}': {delete_error}")

        # Пытаемся обновить и сохранить файл
        success = update_and_save_excel(file_path_shows_expenses, new_file_path)

        if not success:
            # Если все попытки неудачны, выводим сообщение пользователю
            while not success:
                input("Обновить Excel файл не получилось. Закройте все открытые файлы и нажмите любую кнопку для повторной попытки.")
                success = update_and_save_excel(file_path_shows_expenses, new_file_path)

            print("Файл успешно обновлен после повторной попытки.")

        # После успешного обновления копируем файл в папку FOLDER_PATH_DUDL
        if success:
            try:
                shutil.copy(new_file_path, dudl_file_path)
                print(f"Файл успешно скопирован в папку '{FOLDER_PATH_DUDL}'.")
            except Exception as copy_error:
                print(f"Ошибка при копировании файла в папку '{FOLDER_PATH_DUDL}': {copy_error}")

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Файл успешно обновлен и сохранен. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Показы и затраты ОЗ_2.0.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при обработке файла 'Показы и затраты ОЗ_2.0.xlsx': {e}")

Подготовка данных для ДБ завершена.
Начинаем обновлять файл 'Показы и затраты ОЗ_2.0.xlsx'...
Файл 'Показы и затраты ОЗ_2.0 30.09.xlsx' удален из папки '\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ'.
Файл 'Показы и затраты ОЗ_2.0 30.09.xlsx' удален из папки '\\kari.local\public\all\Analytics\Marketplaceanalytics\Дашбоард по рекламным кампаниям'.
Попытка 1 обновить файл 'Показы и затраты ОЗ_2.0.xlsx'...
Выполняем обновление данных...
Файл успешно сохранен с оригинальным именем в '\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям'.
Файл успешно сохранен как 'Показы и затраты ОЗ_2.0 01.10.xlsx'.
Файл успешно скопирован в папку '\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ'.
Файл успешно обновлен и сохранен. Время выполнения: 0 часа(ов) 30 минут(ы) 4.35 секунд
